# OpenCV Crash Course: 2026 Edition
## Module 3: Drawing & Contours
**Estimated time: 20-25 minutes**

---

### What you'll learn
- Draw shapes: lines, rectangles, circles, polygons
- Draw a detection-style label box (the same thing YOLO output looks like)
- Find and draw contours
- Contour properties: area, perimeter, centroid, bounding box

> Drawing + contours = the foundation of **object detection annotation**.

In [ ]:
import cv2
import numpy as np
import matplotlib.pyplot as plt

def show(img, title="", figsize=(8, 6)):
    plt.figure(figsize=figsize)
    if len(img.shape) == 3:
        plt.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    else:
        plt.imshow(img, cmap='gray')
    plt.title(title, fontsize=13)
    plt.axis('off')
    plt.tight_layout()
    plt.show()

def show_grid(images, titles, cols=3, figsize=(15, 5)):
    rows = (len(images) - 1) // cols + 1
    fig, axes = plt.subplots(rows, cols, figsize=figsize)
    axes = np.array(axes).flatten()
    for i, (img, title) in enumerate(zip(images, titles)):
        if len(img.shape) == 3:
            axes[i].imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
        else:
            axes[i].imshow(img, cmap='gray')
        axes[i].set_title(title, fontsize=11)
        axes[i].axis('off')
    for j in range(len(images), len(axes)):
        axes[j].axis('off')
    plt.tight_layout()
    plt.show()

## 1. Drawing on Images

All drawing functions take:
- `img` - the image to draw on (modified **in-place**)
- `color` - in **(B, G, R)** format
- `thickness` - stroke width; `-1` fills the shape

In [ ]:
canvas = np.zeros((500, 700, 3), dtype=np.uint8)

# Line: (img, pt1, pt2, color, thickness)
cv2.line(canvas, (30, 30), (300, 30), (255, 255, 255), 2)
cv2.line(canvas, (30, 60), (300, 60), (0, 255, 0), 4)         # green, thicker

# Rectangle: (img, top-left, bottom-right, color, thickness)
cv2.rectangle(canvas, (50, 100), (200, 200), (0, 100, 255), 2)  # orange outline
cv2.rectangle(canvas, (220, 100), (370, 200), (0, 200, 80), -1) # green filled

# Circle: (img, center, radius, color, thickness)
cv2.circle(canvas, (520, 150), 60, (255, 50, 50), 3)   # blue outline
cv2.circle(canvas, (620, 150), 50, (180, 0, 180), -1)  # purple filled

# Ellipse: (img, center, axes, angle, startAngle, endAngle, color, thickness)
cv2.ellipse(canvas, (150, 310), (100, 60), 0, 0, 360, (0, 200, 255), 2)
cv2.ellipse(canvas, (380, 310), (80, 120), 45, 0, 360, (255, 200, 0), -1)

# Polygon
pts = np.array([[530, 250], [600, 380], [460, 380]], dtype=np.int32)
cv2.polylines(canvas, [pts], isClosed=True, color=(0, 255, 255), thickness=2)

pts_filled = np.array([[530, 400], [620, 480], [440, 480]], dtype=np.int32)
cv2.fillPoly(canvas, [pts_filled], color=(100, 255, 100))

show(canvas, "Drawing Shapes")

## 2. Text Annotations

In [ ]:
# Practical: draw a label box like a detection annotation
annotated = np.zeros((300, 500, 3), dtype=np.uint8)
annotated[:] = (30, 30, 30)

label   = "Person  0.94"
box_pt1 = (50, 40)
box_pt2 = (380, 260)
color   = (0, 200, 80)

# Draw bounding box
cv2.rectangle(annotated, box_pt1, box_pt2, color, 2)

# Label background
(tw, th), baseline = cv2.getTextSize(label, cv2.FONT_HERSHEY_SIMPLEX, 0.7, 2)
cv2.rectangle(annotated, (box_pt1[0], box_pt1[1] - th - 8),
              (box_pt1[0] + tw + 4, box_pt1[1]), color, -1)

# Label text
cv2.putText(annotated, label, (box_pt1[0] + 2, box_pt1[1] - 4),
            cv2.FONT_HERSHEY_SIMPLEX, 0.7, (0, 0, 0), 2)

show(annotated, "Detection-style annotation")

## 3. Finding & Drawing Contours

Contours are curves joining all **continuous points** along a boundary.  
Workflow: **Grayscale → Threshold / Canny → findContours → drawContours**

In [ ]:
# Create a scene with distinct shapes
scene = np.zeros((400, 600, 3), dtype=np.uint8)
cv2.rectangle(scene, (30, 30),   (150, 150),  (255, 255, 255), -1)
cv2.circle(scene,   (280, 90),   70,           (255, 255, 255), -1)
pts_tri = np.array([[450, 30], [570, 160], [330, 160]], dtype=np.int32)
cv2.fillPoly(scene, [pts_tri], (255, 255, 255))
cv2.rectangle(scene, (50, 230),  (220, 370),  (255, 255, 255), -1)
cv2.circle(scene,   (380, 300),  80,           (255, 255, 255), -1)

gray_scene = cv2.cvtColor(scene, cv2.COLOR_BGR2GRAY)

# RETR_EXTERNAL: only outermost contours
# CHAIN_APPROX_SIMPLE: compress horizontal/vertical/diagonal segments
contours, hierarchy = cv2.findContours(gray_scene, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
print(f"Found {len(contours)} contours")

# Draw all contours
drawn = scene.copy()
cv2.drawContours(drawn, contours, -1, (0, 255, 0), 3)  # -1 = draw all

show_grid([scene, drawn], ["Original", "Contours drawn"], cols=2, figsize=(12, 5))

## 4. Contour Properties

In [ ]:
for i, cnt in enumerate(contours):
    area      = cv2.contourArea(cnt)
    perimeter = cv2.arcLength(cnt, closed=True)
    M         = cv2.moments(cnt)
    if M['m00'] != 0:
        cx = int(M['m10'] / M['m00'])  # centroid x
        cy = int(M['m01'] / M['m00'])  # centroid y
    else:
        cx, cy = 0, 0
    print(f"Contour {i}: area={area:.0f}  perimeter={perimeter:.0f}  centroid=({cx},{cy})")

In [ ]:
annotated_scene = scene.copy()

for cnt in contours:
    # Upright bounding rectangle
    x, y, w, h = cv2.boundingRect(cnt)
    cv2.rectangle(annotated_scene, (x, y), (x+w, y+h), (0, 100, 255), 2)

    # Minimum enclosing circle
    (cx, cy), radius = cv2.minEnclosingCircle(cnt)
    cv2.circle(annotated_scene, (int(cx), int(cy)), int(radius), (255, 0, 200), 1)

    # Centroid dot
    M = cv2.moments(cnt)
    if M['m00'] != 0:
        px = int(M['m10'] / M['m00'])
        py = int(M['m01'] / M['m00'])
        cv2.circle(annotated_scene, (px, py), 5, (0, 255, 255), -1)

show(annotated_scene, "Bounding rects + enclosing circles + centroids")

In [ ]:
# Rotated (minimum-area) bounding rectangle
rot_scene = scene.copy()
for cnt in contours:
    rect  = cv2.minAreaRect(cnt)
    box   = cv2.boxPoints(rect).astype(np.int32)
    cv2.drawContours(rot_scene, [box], 0, (0, 255, 0), 2)

show_grid([annotated_scene, rot_scene],
          ["Upright bounding rect", "Minimum-area rotated rect"],
          cols=2, figsize=(12, 5))

---

## Module 3 Summary

| Task | Code |
|------|------|
| Line | `cv2.line(img, pt1, pt2, color, thickness)` |
| Rectangle | `cv2.rectangle(img, pt1, pt2, color, -1)` |
| Circle | `cv2.circle(img, center, radius, color, -1)` |
| Text | `cv2.putText(img, text, org, font, scale, color, thick)` |
| Find contours | `cv2.findContours(binary, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)` |
| Area | `cv2.contourArea(cnt)` |
| Bounding box | `cv2.boundingRect(cnt)` → x, y, w, h |
| Rotated box | `cv2.minAreaRect(cnt)` |

**Next:** Module 4 - Video & Real-time Processing